# Piece 2 — joint sentence + token head (Kaggle)

`loss = token_loss + lambda * sentence_loss`, sweeping lambda over
0, 0.1, 0.3, 0.5, 1.0.

**Lambda 0 is the control.** At lambda 0 the script builds the model with no
sentence head at all, so that row is the Piece 1 architecture exactly and this
experiment stands on its own.

## How to run

1. Settings: **Accelerator = GPU T4 x2**, **Internet = On**
2. Set `OWNER` below
3. Leave `SMOKE = True`, **Run All** (~5 min) — confirms nothing crashes
4. Set `SMOKE = False`
5. **Save Version -> Save & Run All (Commit)**, then close the browser
6. When it finishes, the two result files are in that version's **Output** tab

Test is never scored here. Everything is chosen on validation.

## 1. Settings

In [ ]:
OWNER  = 'YOUR_NAME'     # <-- EDIT: recorded in every results row
SMOKE  = True            # True = 1 seed / 3 epochs check. Set False for the real run.

REPO   = 'https://github.com/hatheem-r/project_DNN.git'
BRANCH = 'main'

print('owner', OWNER, '| smoke', SMOKE)

## 2. GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

## 3. Code

Needs Internet = On in Settings.

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
print('cwd', os.getcwd())
!ls src/ notebooks/

## 4. fastText vectors

About 460 MB. It goes in `/kaggle/temp`, which is **not** saved as notebook
output, so it does not bloat the version you commit.

The script reads the path from the `SOLD_VECTORS` environment variable, so
there is nothing to pass on the command line.

In [ ]:
import os
os.makedirs('/kaggle/temp/embeddings', exist_ok=True)
VEC = '/kaggle/temp/embeddings/cc.si.300.vec.gz'
if not os.path.exists(VEC):
    !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
os.environ['SOLD_VECTORS'] = VEC
os.environ['OWNER'] = OWNER
!ls -lh $VEC

## 5. Safety tests — do not skip

The alignment tests catch a bug that would silently shift labels against words
while the loss still falls and nothing crashes.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## 6. The run

5 seeds, not the script default of 3, because the project rule is that every
reported number is a mean over 5 seeds with its standard deviation. ~2.5 h.

`--loss` is left at its default (`cross_entropy`).

**Read precision and recall, not just F1.** After Piece 1 the model sits near
P 0.74 / R 0.68, close to balanced, so there is little headroom. A lambda that
lifts recall while collapsing precision is a net loss.

**Expect the lambda 0 row near 0.704**, not the 0.7083 quoted for Piece 1.
Piece 1's headline used CRF on / batch 32; this runs CRF off / batch 64. Same
architecture, different training config. Compare every lambda against this
run's own lambda 0 row.

A null is the likely outcome. Report the best lambda regardless — Piece 4
cannot run without a sentence head, because SemiSOLD's teacher scores are
sentence level and distillation has nothing to attach to otherwise.

In [ ]:
import os, subprocess
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

args = ['python', 'notebooks/09_pieces_234.py', '--piece2']
args += (['--seeds', '1', '--epochs', '3', '--patience', '2'] if SMOKE
         else ['--seeds', '1', '2', '3', '4', '5'])
print(' '.join(args), flush=True)

with open('results/piece2_report.txt', 'w') as log:
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        log.write(line); log.flush()
    rc = p.wait()
print('
exit code', rc)

## 7. Save the output

In [ ]:
import os, shutil

if SMOKE:
    # the smoke run appended junk rows - drop them so the real run starts clean
    if os.path.exists('results/results_piece2.csv'):
        os.remove('results/results_piece2.csv')
    print('smoke rows cleared.')
    print('Now set SMOKE = False and use Save Version -> Save & Run All (Commit).')
else:
    for f in ('results/results_piece2.csv', 'results/piece2_report.txt'):
        shutil.copy(f, '/kaggle/working/')
    print('saved to /kaggle/working:')
    for f in sorted(os.listdir('/kaggle/working')):
        if f.startswith('piece2') or f.startswith('results_piece2'):
            print('  ', f, os.path.getsize('/kaggle/working/' + f), 'bytes')

## 8. Report to the group

- lambda = 0 (control): F1 ± std
- best lambda, its F1 ± std, and its P / R
- whether it beat lambda = 0 (the script prints the verdict)
- the lambda value Piece 4 should use — needed even if the gain is null

Download `results_piece2.csv` and `piece2_report.txt` from the Output tab, drop
them into `results/` in the repo, then:

```
git add results/results_piece2.csv results/piece2_report.txt
git commit -m "piece2 results"
git pull --rebase
git push
```